In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
import joblib
from pathlib import Path

DATA_PATH = Path("data/sentimen_dataset.xlsx")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)
MODEL_PATH = MODEL_DIR / "svm_pipeline.pkl"

In [5]:
df = pd.read_excel(DATA_PATH)
df.head()

,Text,Label
0,"Video ini sangat membantu, poin ke-1 dijelaska...",Positif
1,Saya kurang puas dengan penjelasan pada bagian...,Negatif
2,Saya menonton video ini sampai menit ke-1,Netral
3,"Video ini sangat membantu, poin ke-2 dijelaska...",Positif
4,Saya kurang puas dengan penjelasan pada bagian...,Negatif


In [7]:
df = df.dropna(subset=["Text", "Label"])
df["Text"] = df["Text"].astype(str)
df["Label"] = df["Label"].astype(str)

df["Label"].value_counts()

Label
Netral     170
Positif    167
Negatif    167
Name: count, dtype: int64

In [9]:
X = df["Text"]
y = df["Label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y  # supaya proporsi label seimbang
)
len(X_train), len(X_test)

(403, 101)

In [10]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),      # unigram + bigram
    )),
    ("svm", SVC(
        kernel="linear",
        probability=True,        # penting supaya bisa ambil probability
    )),
])

In [11]:
print("Training SVM...")
pipeline.fit(X_train, y_train)
print("Training selesai.")

Training SVM...
Training selesai.


In [12]:
y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

     Negatif       1.00      1.00      1.00        34
      Netral       1.00      1.00      1.00        34
     Positif       1.00      1.00      1.00        33

    accuracy                           1.00       101
   macro avg       1.00      1.00      1.00       101
weighted avg       1.00      1.00      1.00       101



In [13]:
joblib.dump(pipeline, MODEL_PATH)
print(f"Model disimpan ke {MODEL_PATH}")

Model disimpan ke models\svm_pipeline.pkl
